# NewsMood — 02. Évaluation des modèles de sentiment

**Objectif :** mesurer avec des chiffres quel modèle est le meilleur, TextBlob ou FinBERT.

**Deux jeux de données annotés à la main :**
- **Financial PhraseBank** (2 264 phrases, annotateurs unanimes) — ⚠️ FinBERT a été entraîné dessus : risque de *fuite de données* (score gonflé).
- **Twitter Financial News** (partie validation) — jamais vu par FinBERT : c'est le vrai test.

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from datasets import load_dataset

# --- 1. Financial PhraseBank -----------------------------------------------
# Copie publiee par le FinTech Lab de Georgia Tech (benchmark FLaME).
# Elle est coupee en "train" (environ 1 580 phrases) et "test" (environ 680 phrases) :
# on recolle les deux pour retrouver les 2 264 phrases.
fpb_parts = load_dataset("gtfintechlab/financial_phrasebank_sentences_allagree", "5768")
fpb = pd.concat([fpb_parts["train"].to_pandas(), fpb_parts["test"].to_pandas()], ignore_index=True)
fpb = fpb.rename(columns={"sentence": "text"})[["text", "label"]]
fpb["label"] = fpb["label"].map({0: "negative", 1: "neutral", 2: "positive"})

# --- 2. Twitter Financial News (partie validation) ------------------------
tw = load_dataset("zeroshot/twitter-financial-news-sentiment")["validation"].to_pandas()
tw["label"] = tw["label"].map({0: "negative", 1: "positive", 2: "neutral"})  # Bearish / Bullish / Neutral

# --- On regarde ------------------------------------------------------------
for nom, df in [("Financial PhraseBank", fpb), ("Twitter Financial News", tw)]:
    print(f"=== {nom} : {len(df)} textes ===")
    print((df["label"].value_counts(normalize=True) * 100).round(1).to_string())
    print()

print("Exemples PhraseBank :")
print(fpb.sample(3, random_state=42).to_string(index=False))
print("\nExemples Twitter :")
print(tw.sample(3, random_state=42).to_string(index=False))

c:\Users\Dell\projets\newsmood\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\datasets--gtfintechlab--financial_phrasebank_sentences_allagree. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Dell\projets\newsmood\venv\Lib\site-packages\huggingface_hub\file_down

=== Financial PhraseBank : 2264 textes ===
label
neutral     61.4
positive    25.2
negative    13.4

=== Twitter Financial News : 2388 textes ===
label
neutral     65.6
positive    19.9
negative    14.5

Exemples PhraseBank :
                                                                                    text    label
The acquisition will have an immediate positive impact on Aspocomp 's financial result . positive
                             Sales of security and system packaging increased slightly . positive
                    - Counter your competitor 's strengths and target their weaknesses .  neutral

Exemples Twitter :
                                                                                                                                        text   label
Two key regulators are holding off on easing Wall Street debt limits, leaving billions of dollars locked up that co… https://t.co/YvDDlEcmh2 neutral
            Xinjiang Xinxin Mining Industry : ANNOUNCEMENT - RES

## 1. Scores des modèles

Métrique principale : **F1-score macro** (les classes sont déséquilibrées : 61 à 66 % de neutres).
Référence (*baseline*) : un modèle qui répond toujours « neutral ».

In [5]:
import time
from sklearn.metrics import accuracy_score, f1_score
from src.sentiment import textblob_predict, finbert_predict

LABELS = ["negative", "neutral", "positive"]

def evaluer(df, nom_jeu):
    """Fait predire les 3 modeles sur df, ajoute les predictions a df, renvoie un tableau de scores."""
    textes = df["text"].tolist()
    y_true = df["label"].tolist()
    lignes = []

    modeles = {
        "Toujours neutre": lambda t: [{"label": "neutral"}] * len(t),
        "TextBlob": textblob_predict,
        "FinBERT": finbert_predict,
    }
    for nom_modele, predire in modeles.items():
        debut = time.time()
        y_pred = [r["label"] for r in predire(textes)]
        duree = time.time() - debut

        df[f"pred_{nom_modele}"] = y_pred   # on garde les predictions pour l'analyse des erreurs
        lignes.append({
            "jeu": nom_jeu,
            "modele": nom_modele,
            "accuracy": round(accuracy_score(y_true, y_pred), 3),
            "f1_macro": round(f1_score(y_true, y_pred, labels=LABELS, average="macro"), 3),
            "temps_s": round(duree, 1),
        })
        print(f"{nom_jeu} - {nom_modele} : termine en {duree:.0f} s")
    return pd.DataFrame(lignes)

scores = pd.concat([evaluer(fpb, "PhraseBank"), evaluer(tw, "Twitter")], ignore_index=True)
scores

PhraseBank - Toujours neutre : termine en 0 s
PhraseBank - TextBlob : termine en 1 s


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 164.02it/s]


PhraseBank - FinBERT : termine en 185 s
Twitter - Toujours neutre : termine en 0 s
Twitter - TextBlob : termine en 0 s
Twitter - FinBERT : termine en 173 s


,jeu,modele,accuracy,f1_macro,temps_s
0,PhraseBank,Toujours neutre,0.614,0.254,0.0
1,PhraseBank,TextBlob,0.543,0.467,1.0
2,PhraseBank,FinBERT,0.972,0.963,185.0
3,Twitter,Toujours neutre,0.656,0.264,0.0
4,Twitter,TextBlob,0.503,0.385,0.5
5,Twitter,FinBERT,0.725,0.668,172.8
